In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)

feedback_source_str = "feedback"
feedback_source_tn = Tn.source(feedback_source_str)

input_with_window_end_tn = (
    order_source_tn
    .map(lambda x: x | {"window_end": x["value"]["ts"] + 10})
)

combined_input_with_window_end_tn = (
    input_with_window_end_tn
    .merge(feedback_source_tn)
)

input_cur_ts_tn = (
    combined_input_with_window_end_tn
    .map(lambda x: {"ts": x["value"]["ts"]})
    .max(lambda x: x["ts"],
         lambda x: {"cur_ts": x})
)

join_window_end_tn = (
    combined_input_with_window_end_tn
    .join(input_cur_ts_tn,
          lambda l, r: r["cur_ts"] > l["window_end"],
          lambda l, _: l)
    ._filter(lambda _, w: w > 0)
    ._neg()
    ._delay()
)
built_tn = (
    join_window_end_tn
    .merge(input_with_window_end_tn)
    ._peek("root")
)
built_tn.build()
print(built_tn.mermaid())
order_source_tn.to_zSet(Tn._from_records)
feedback_source_tn.to_zSet(Tn._from_records)
join_window_end_tn.from_zSet(Tn._to_records)
built_tn.from_zSet(Tn._to_records)
#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
built_tn.push({order_source_str: [(gen_order(1, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 2")
built_tn.push({order_source_str: [(gen_order(2, 11), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 3")
built_tn.push({order_source_str: [(gen_order(3, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()
print(len(pickle.dumps(built_tn)))
built_tn.push({order_source_str: [], feedback_source_str: join_window_end_tn.latest()})
built_tn.latest()

for i in range(1000):
    print(i)
    built_tn.push({order_source_str: [(gen_order(i + 4, i * 10 + 11), 1)], feedback_source_str: join_window_end_tn.latest()})
    built_tn.latest()
    print(len(pickle.dumps(built_tn)))


In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

source1_str = "source_1"
source2_str = "source_2"

source1_tn = Tn.source(source1_str)
source1_tn.to_zSet(Tn._from_records)
source2_tn = Tn.source(source2_str)
source2_tn.to_zSet(Tn._from_records)

built_tn = source1_tn.merge(source2_tn)._peek()
built_tn.from_zSet(Tn._to_records)

built_tn.build()

built_tn.push(source1_str, [("bla", 1)])
built_tn.latest()
built_tn.push(source2_str, [("bla", -1)])
built_tn.latest()

In [4]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)

expire_tn = order_source_tn.expire(
     lambda x: x["value"]["ts"],
     lambda x: x["value"]["ts"] + 10,
     lambda x: (x[0]["value"].__setitem__("expiry", x[1]) or x[0])
)._peek()

built_tn = Tn.build(expire_tn)

built_tn.from_zSet(Tn._to_records)

#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
built_tn.push(order_source_str, [(gen_order(1, 0), 1)])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 2")
built_tn.push(order_source_str, [(gen_order(2, 11), 1)])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
print("Step 3")
built_tn.push(order_source_str, [(gen_order(3, 0), 1)])
built_tn.latest()
print(len(pickle.dumps(built_tn)))
built_tn.push(order_source_str, [])
built_tn.latest()

for i in range(1000):
    print(i)
    built_tn.push(order_source_str, [(gen_order(i + 4, i * 10 + 11), 1)])
    built_tn.latest()
    print(len(pickle.dumps(built_tn)))


Step 1
({'value': {'order_id': 1, 'ts': 0, 'expiry': 10}}, 1)
26499
Step 2
({'value': {'order_id': 2, 'ts': 11, 'expiry': 21}}, 1)
27412
Step 3
({'value': {'order_id': 1, 'ts': 0, 'expiry': 10}}, -1)
({'value': {'order_id': 3, 'ts': 0, 'expiry': 10}}, 1)
28267
({'value': {'order_id': 3, 'ts': 0, 'expiry': 10}}, -1)
0
({'value': {'order_id': 4, 'ts': 11, 'expiry': 21}}, 1)
28975
1
({'value': {'order_id': 5, 'ts': 21, 'expiry': 31}}, 1)
29013
2
({'value': {'order_id': 6, 'ts': 31, 'expiry': 41}}, 1)
29049
3
({'value': {'order_id': 4, 'ts': 11, 'expiry': 21}}, -1)
({'value': {'order_id': 2, 'ts': 11, 'expiry': 21}}, -1)
({'value': {'order_id': 7, 'ts': 41, 'expiry': 51}}, 1)
29222
4
({'value': {'order_id': 5, 'ts': 21, 'expiry': 31}}, -1)
({'value': {'order_id': 8, 'ts': 51, 'expiry': 61}}, 1)
29360
5
({'value': {'order_id': 6, 'ts': 31, 'expiry': 41}}, -1)
({'value': {'order_id': 9, 'ts': 61, 'expiry': 71}}, 1)
29437
6
({'value': {'order_id': 7, 'ts': 41, 'expiry': 51}}, -1)
({'value': {